# 6BUIS017W CWK1 — Market Analysis and Stock Segmentation Using RFM

In [1]:
!git clone https://github.com/aaisha-aamina/crm-sp500-rfm-segmentation
%cd crm-sp500-rfm-segmentation

Cloning into 'crm-sp500-rfm-segmentation'...
remote: Enumerating objects: 20, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 20 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (20/20), 13.59 MiB | 27.44 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/crm-sp500-rfm-segmentation


# Task 1 — Data Preparation and Feature Engineering
## Setup and configuration
`USE_SNAPSHOT = True` makes the notebook read only the saved files in `data/`. Set it to `False` **only** to repeat the one-off download (this would overwrite the snapshot and can change results, because adjusted prices and the constituent list are revised over time).

In [2]:
import io, json, sys, platform
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
import requests
import yfinance as yf

START, END = '2021-01-01', '2025-01-01'     # END is exclusive
BENCH = '^GSPC'
USE_SNAPSHOT = True    # False = download now; True = only read saved files (use this on all later runs)

DATA_DIR = Path('data'); DATA_DIR.mkdir(exist_ok=True)
F_CONST  = DATA_DIR / 'sp500_constituents.csv'
F_HTML   = DATA_DIR / 'sp500_wikipedia_snapshot.html'
F_PRICES = DATA_DIR / 'raw_prices_long.csv.gz'
F_META   = DATA_DIR / 'metadata.json'

versions = {'python': sys.version.split()[0], 'pandas': pd.__version__,
            'numpy': np.__version__, 'yfinance': yf.__version__}
print(versions)

{'python': '3.13.18', 'pandas': '2.2.3', 'numpy': '2.4.6', 'yfinance': '0.2.66'}


## 1.1 Ticker selection and data acquisition
* **Ticker list:** current S&P 500 constituents read from Wikipedia with `pandas.read_html`; the retrieval date is stored in `metadata.json`. Wikipedia writes share classes as `BRK.B`; Yahoo uses `BRK-B`, so dots are replaced by hyphens.
* **Price source (documented choice):** `yfinance` (Yahoo Finance) for daily Open, High, Low, Close, Adjusted Close and Volume, with `auto_adjust=False` so that raw Close (needed for Monetary) and Adjusted Close (needed for returns) are both kept. The benchmark is `^GSPC`.
* **Period:** 1 Jan 2021 to 1 Jan 2025, end date exclusive.

In [3]:
WIKI_URL = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'

if USE_SNAPSHOT:
    cons = pd.read_csv(F_CONST)
    meta = json.loads(F_META.read_text())
else:
    r = requests.get(WIKI_URL, headers={'User-Agent': 'Mozilla/5.0 (university coursework 6BUIS017W)'}, timeout=30)
    r.raise_for_status()
    F_HTML.write_text(r.text, encoding='utf-8')
    cons = pd.read_html(io.StringIO(r.text))[0]
    # Wikipedia writes BRK.B, Yahoo writes BRK-B
    cons['yf_ticker'] = cons['Symbol'].astype(str).str.strip().str.replace('.', '-', regex=False)
    cons.to_csv(F_CONST, index=False)
    now = datetime.now(timezone.utc)
    meta = {'wikipedia_url': WIKI_URL,
            'retrieval_date_utc': str(now.date()),
            'retrieval_timestamp_utc': now.isoformat(timespec='seconds'),
            'price_source': 'yfinance (auto_adjust=False)',
            'versions': versions, 'start': START, 'end_exclusive': END, 'benchmark': BENCH}
    F_META.write_text(json.dumps(meta, indent=2))

assert not cons['yf_ticker'].duplicated().any()
print(len(cons), 'tickers | retrieved:', meta['retrieval_date_utc'])
cons[['Symbol', 'yf_ticker', 'Security', 'GICS Sector']].head()

503 tickers | retrieved: 2026-11-14


,Symbol,yf_ticker,Security,GICS Sector
0,MMM,MMM,3M,Industrials
1,AOS,AOS,A. O. Smith,Industrials
2,ABT,ABT,Abbott Laboratories,Health Care
3,ABBV,ABBV,AbbVie,Health Care
4,ACN,ACN,Accenture,Information Technology


In [4]:
FIELDS = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']

def download_batch(tick_list):
    d = yf.download(tick_list, start=START, end=END, auto_adjust=False, progress=False, threads=True)
    if d is None or d.empty:
        return None
    frames = []
    for f in FIELDS:
        w = d[f].copy(); w.index.name = 'Date'
        frames.append(w.melt(ignore_index=False, var_name='Ticker', value_name=f)
                       .reset_index().set_index(['Date', 'Ticker']))
    return pd.concat(frames, axis=1).reset_index()

if not USE_SNAPSHOT:
    tickers = cons['yf_ticker'].tolist() + [BENCH]
    parts = []
    for i in range(0, len(tickers), 50):
        got = download_batch(tickers[i:i + 50])
        if got is not None:
            parts.append(got)
        print(f'batch {i // 50 + 1}/{-(-len(tickers) // 50)} done')
    long_df = pd.concat(parts, ignore_index=True)

    missing = sorted(set(tickers) - set(long_df['Ticker'].unique()))
    if missing:
        print('retrying:', missing)
        retry = download_batch(missing if len(missing) > 1 else missing + [BENCH])
        if retry is not None:
            long_df = pd.concat([long_df, retry[retry['Ticker'].isin(missing)]], ignore_index=True)

    long_df.to_csv(F_PRICES, index=False, compression='gzip')
    meta['price_download_timestamp_utc'] = datetime.now(timezone.utc).isoformat(timespec='seconds')
    F_META.write_text(json.dumps(meta, indent=2))
    print('saved', F_PRICES, long_df.shape)

**Snapshot check.** From here on the analysis always reads the saved file, never the live download. Shares with no data returned and shares with fewer rows than the benchmark are listed below and carried into the exclusion log in 1.2.

In [6]:
raw = pd.read_csv(F_PRICES, parse_dates=['Date'])
print('rows:', raw.shape[0], '| dates:', raw['Date'].min().date(), '->', raw['Date'].max().date())
print('tickers in file:', raw['Ticker'].nunique(), '| benchmark present:', BENCH in raw['Ticker'].values)
print('file size (MB):', round(F_PRICES.stat().st_size / 1e6, 1))

listed = set(cons['yf_ticker'])
got = set(raw['Ticker'])
failed = sorted(listed - got)
print('listed but NO data returned:', failed)
pd.DataFrame({'ticker': failed, 'reason': 'no data returned by yfinance'}).to_csv(DATA_DIR / 'failed_downloads.csv', index=False)

cov = raw.groupby('Ticker')['Date'].agg(first_date='min', last_date='max', n_obs='count')
cov.to_csv(DATA_DIR / 'download_coverage.csv')
print('shares with fewer rows than the benchmark:', int((cov['n_obs'] < cov.loc[BENCH, 'n_obs']).sum()))
cov.sort_values('n_obs').head()

rows: 506520 | dates: 2021-01-04 -> 2024-12-31
tickers in file: 504 | benchmark present: True
file size (MB): 13.1
listed but NO data returned: []
shares with fewer rows than the benchmark: 0


,first_date,last_date,n_obs
Ticker,,,
A,2021-01-04,2024-12-31,1005
AAPL,2021-01-04,2024-12-31,1005
ABBV,2021-01-04,2024-12-31,1005
ABNB,2021-01-04,2024-12-31,1005
ABT,2021-01-04,2024-12-31,1005


## 1.2 Data validation and alignment
Checks: duplicate dates, missing values, non-positive prices, volume errors (missing, zero, negative), unequal histories against the benchmark trading calendar, and corporate actions. **No missing price is ever filled.** A share failing any check is excluded and written to the exclusion log, so every exclusion is explained.

In [7]:
OUT_DIR = Path('outputs'); OUT_DIR.mkdir(exist_ok=True)
RFM_BASE, RFM_WIN = 20, 60        # used later in 1.4; needed here to flag splits inside the RFM window

# --- duplicates ---
n_dup = int(raw.duplicated(['Date', 'Ticker']).sum())
print('duplicate (Date, Ticker) rows:', n_dup)
raw = raw.drop_duplicates(['Date', 'Ticker'], keep='first')

adj   = raw.pivot(index='Date', columns='Ticker', values='Adj Close').sort_index()
close = raw.pivot(index='Date', columns='Ticker', values='Close').sort_index()
vol   = raw.pivot(index='Date', columns='Ticker', values='Volume').sort_index()

# --- benchmark calendar = the reference set of trading dates ---
cal = adj[BENCH].dropna().index
print('benchmark trading days:', len(cal), '|', cal.min().date(), '->', cal.max().date())
assert cal.min() >= pd.Timestamp(START) and cal.max() < pd.Timestamp(END)
extra = adj.index.difference(cal)
print('dates present for shares but not in benchmark calendar:', len(extra))

# verify (don't assume) that the index has Adj Close = Close
print('^GSPC Adj Close equals Close on all dates:', np.allclose(adj.loc[cal, BENCH], close.loc[cal, BENCH]))

adj_c, close_c, vol_c = adj.reindex(cal), close.reindex(cal), vol.reindex(cal)

duplicate (Date, Ticker) rows: 0
benchmark trading days: 1005 | 2021-01-04 -> 2024-12-31
dates present for shares but not in benchmark calendar: 0
^GSPC Adj Close equals Close on all dates: True


Per-share validation. A share is retained only if it passes every check.

In [8]:
cand = cons['yf_ticker'].tolist()
reasons = {}
def add(t, msg): reasons.setdefault(t, []).append(msg)

present = [t for t in cand if t in adj_c.columns]
stats = pd.DataFrame(index=cand)
stats['in_download']      = [t in adj_c.columns for t in cand]
stats['n_obs']            = adj_c[present].notna().sum().reindex(cand)
stats['n_missing_adj']    = adj_c[present].isna().sum().reindex(cand)
stats['n_missing_close']  = close_c[present].isna().sum().reindex(cand)
stats['n_missing_vol']    = vol_c[present].isna().sum().reindex(cand)
stats['n_nonpos_price']   = ((close_c[present] <= 0) | (adj_c[present] <= 0)).sum().reindex(cand)
stats['n_zero_or_neg_vol'] = (vol_c[present] <= 0).sum().reindex(cand)
stats['first_date']       = adj_c[present].apply(lambda s: s.first_valid_index()).reindex(cand)

for t in cand:
    s = stats.loc[t]
    if not s['in_download'] or s['n_obs'] == 0:
        add(t, 'no data returned by yfinance'); continue
    if s['n_obs'] < len(cal):
        add(t, f"unequal history: {int(s['n_obs'])}/{len(cal)} benchmark days (first date {pd.Timestamp(s['first_date']).date()})")
    if s['n_missing_close'] > 0 or s['n_missing_adj'] > 0:
        add(t, f"missing prices (Close {int(s['n_missing_close'])}, Adj {int(s['n_missing_adj'])})")
    if s['n_missing_vol'] > 0:
        add(t, f"missing volume on {int(s['n_missing_vol'])} days")
    if s['n_nonpos_price'] > 0:
        add(t, f"non-positive price on {int(s['n_nonpos_price'])} days")
    if s['n_zero_or_neg_vol'] > 0:
        add(t, f"zero/negative volume on {int(s['n_zero_or_neg_vol'])} days")

retained = [t for t in cand if t not in reasons]
info = cons.set_index('yf_ticker')
excl = pd.DataFrame([{'ticker': t, 'security': info.loc[t, 'Security'],
                      'sector': info.loc[t, 'GICS Sector'], 'reason': '; '.join(r)}
                     for t, r in reasons.items()],
                    columns=['ticker', 'security', 'sector', 'reason'])
print(f'Constituents: {len(cand)} | retained: {len(retained)} | excluded: {len(excl)}')
excl.to_csv(OUT_DIR / 'exclusion_log.csv', index=False)
stats.to_csv(OUT_DIR / 'validation_summary.csv')
excl

Constituents: 503 | retained: 486 | excluded: 17


,ticker,security,sector,reason
0,APP,AppLovin,Communication Services,unequal history: 935/1005 benchmark days (firs...
1,COIN,Coinbase,Financials,unequal history: 936/1005 benchmark days (firs...
2,CEG,Constellation Energy,Utilities,unequal history: 742/1005 benchmark days (firs...
3,EXE,Expand Energy,Energy,unequal history: 979/1005 benchmark days (firs...
4,FDXF,FedEx Freight,Industrials,no data returned by yfinance
5,FERG,Ferguson Enterprises,Industrials,zero/negative volume on 27 days
6,GEHC,GE HealthCare,Health Care,unequal history: 513/1005 benchmark days (firs...
7,GEV,GE Vernova,Industrials,unequal history: 193/1005 benchmark days (firs...
8,HONA,Honeywell Aerospace,Industrials,no data returned by yfinance
9,Q,Qnity Electronics,Information Technology,no data returned by yfinance


**Corporate actions.** Splits and dividends are detected from the gap between raw Close and Adjusted Close: a split changes raw Close but not Adjusted Close.

In [9]:
# A split changes raw Close but not Adj Close, so implied is about the split ratio; normally it is about 1.
cr = close_c[retained].pct_change(fill_method=None)
ar = adj_c[retained].pct_change(fill_method=None)
implied = (1 + ar) / (1 + cr)
is_split = (implied > 1.4) | (implied < 0.7)
split_rows = [(t, d, float(implied.loc[d, t])) for t in retained for d in implied.index[is_split[t]]]
splits = pd.DataFrame(split_rows, columns=['ticker', 'date', 'implied_ratio'])

win_start = cal[-(RFM_BASE + RFM_WIN)]       # earliest session used by the RFM calculation
splits['in_rfm_window'] = splits['date'] >= win_start
print('split-like events in retained shares:', len(splits))
print('...inside the RFM window (last', RFM_BASE + RFM_WIN, 'sessions):', int(splits['in_rfm_window'].sum()))
splits.to_csv(OUT_DIR / 'split_events_detected.csv', index=False)

# how much dividend/split adjustment exists at the start (1.0 = none)
div_factor = (adj_c[retained] / close_c[retained]).iloc[0]
print(div_factor.describe().round(4))
splits.sort_values('date').tail(20)

split-like events in retained shares: 0
...inside the RFM window (last 80 sessions): 0
count    486.0000
mean       0.9027
std        0.0783
min        0.6460
25%        0.8463
50%        0.9087
75%        0.9732
max        1.0000
Name: 2021-01-04 00:00:00, dtype: float64


,ticker,date,implied_ratio,in_rfm_window


**Survivorship bias.** The constituent list is today's membership applied to 2021–2024. Companies removed since 2021 (bankruptcy, acquisition, demotion) are missing, recent additions enter with already-strong histories, and requiring a full 2021–2024 history also drops recent listings and spin-offs. Average returns are therefore biased upward and the sample is not "the S&P 500 of 2021".

**Corporate-action check (NVDA, 10-for-1 split on 10 June 2024).** If Close were not split-adjusted it would fall from about \$1,200 to about \$120 across the split. It does not, so Yahoo's Close is already split-adjusted, and Adjusted Close differs from it only by the dividend adjustment. Volume is on the same split basis, so Close × Volume is consistent.

In [10]:
pd.DataFrame({'Close': close_c['NVDA'], 'Adj Close': adj_c['NVDA'], 'Volume': vol_c['NVDA']}).loc['2024-06-05':'2024-06-12']

,Close,Adj Close,Volume
Date,,,
2024-06-05,122.440002,122.091873,528402000.0
2024-06-06,120.998001,120.653969,664696000.0
2024-06-07,120.888000,120.544289,412386000.0
2024-06-10,121.790001,121.443718,313434100.0
2024-06-11,120.910004,120.576126,222551200.0
2024-06-12,125.199997,124.854271,299595000.0


**Provenance and data-quality evidence.** File checksum, price sanity checks and confirmation that every retained share has the full benchmark history.

In [11]:
import hashlib
# --- 1.1 provenance ---
print('SHA-256 of raw_prices_long.csv.gz:', hashlib.sha256(F_PRICES.read_bytes()).hexdigest()[:20], '...')

# --- 1.2 price sanity checks on retained shares ---
rawr = raw[raw['Ticker'].isin(retained)]
tol = 1e-6
print('High < Low rows:', int((rawr['High'] < rawr['Low']).sum()))
print('Close outside [Low, High] rows:', int(((rawr['Close'] < rawr['Low']*(1-tol)) | (rawr['Close'] > rawr['High']*(1+tol))).sum()))
dret = adj_c[retained].pct_change(fill_method=None)
print('daily |return| > 50% (share-days):', int((dret.abs() > 0.5).sum().sum()))
print('all retained shares have all', len(cal), 'benchmark dates:', bool(adj_c[retained].notna().all().all()))

SHA-256 of raw_prices_long.csv.gz: 4e773f008ff9e199dc14 ...
High < Low rows: 0
Close outside [Low, High] rows: 0
daily |return| > 50% (share-days): 3
all retained shares have all 1005 benchmark dates: True


## 1.3 Weekly return and weekly beta
* Friday-ending weeks, using the final available trading price in each week, with common week labels for every share and the benchmark.
* Incomplete first and last calendar weeks are dropped; holiday-shortened weeks are kept.
* **Weekly return** = adjusted close at the end of the week ÷ adjusted close at the end of the preceding week − 1. Returns are only formed between consecutive weeks, so a multi-week change is never treated as a one-week return.
* **Weekly beta** = Cov(share, benchmark) ÷ Var(benchmark) over all aligned weekly observations (sample covariance and variance, same `ddof`).

In [12]:
# Panel of daily adjusted closes for retained shares + benchmark, on the benchmark calendar
panel = adj_c[retained].copy()
panel[BENCH] = adj_c[BENCH]
assert panel.notna().all().all(), 'unexpected missing values in retained panel'

# Friday-ending weeks, last available trading price in each week
wk = panel.resample('W-FRI').last()

# Drop incomplete first/last calendar weeks; keep holiday-shortened weeks
first_d, last_d = cal.min(), cal.max()
wk_end = wk.index
wk_start = wk_end - pd.Timedelta(days=4)                 # Monday of that week
complete = (wk_start >= first_d) & (wk_end <= last_d)
print('weeks dropped as incomplete:', [str(d.date()) for d in wk_end[~complete]])
wk = wk[complete]

# make sure there are no gaps: consecutive weeks only, so no multi-week change is ever called a 1-week return
assert (np.diff(wk.index.values).astype('timedelta64[D]').astype(int) == 7).all(), 'gap in weekly index'

# weekly return = this week's adj close / previous week's adj close - 1
rets = wk.pct_change().iloc[1:]
print('complete weeks:', len(wk), '| weekly return observations:', len(rets),
      '|', rets.index.min().date(), '->', rets.index.max().date())
print('NaNs in weekly returns:', int(rets.isna().sum().sum()))

weeks dropped as incomplete: ['2025-01-03']
complete weeks: 208 | weekly return observations: 207 | 2021-01-15 -> 2024-12-27
NaNs in weekly returns: 0


Mean weekly return, beta, and an independent regression-slope check of beta.

In [13]:
b = rets[BENCH]
mean_ret = rets[retained].mean()                          # arithmetic mean weekly return
beta = rets[retained].apply(lambda s: s.cov(b)) / b.var() # cov(share, benchmark) / var(benchmark)

# independent check: OLS slope computed a different way
chk = pd.Series({t: np.polyfit(b.values, rets[t].values, 1)[0] for t in retained})
print('max |beta - OLS slope|:', float((beta - chk).abs().max()))
assert (beta - chk).abs().max() < 1e-8

print('weekly observations per share:', len(rets))
print(pd.DataFrame({'mean_weekly_return': mean_ret, 'beta': beta}).describe().round(5))
print('\nlargest absolute weekly returns (sanity check for data errors):')
print(rets[retained].abs().stack().sort_values(ascending=False).head(5).round(3))

max |beta - OLS slope|: 1.9984014443252818e-15
weekly observations per share: 207
       mean_weekly_return       beta
count           486.00000  486.00000
mean              0.00286    0.97370
std               0.00265    0.40734
min              -0.00447    0.05545
25%               0.00118    0.66660
50%               0.00262    0.95061
75%               0.00421    1.17954
max               0.01777    3.77856

largest absolute weekly returns (sanity check for data errors):
Date        Ticker
2023-02-03  CVNA      0.860
2024-11-22  SMCI      0.784
2022-08-05  CVNA      0.612
2024-11-15  BE        0.605
2023-01-13  CVNA      0.592
dtype: float64


## 1.4 Adapted RFM
The observation date is the final common completed trading date. Frequency and Monetary use the latest 60 trading sessions ending on that date. A **high-volume day** has volume strictly greater than twice the mean volume of the **preceding 20 sessions, excluding the day tested**, computed per share and per day (rolling mean shifted by one day).

* **R** — calendar days from the latest high-volume day to the observation date (0 if on that date; missing, never zero, if no event).
* **F** — number of high-volume days in the latest 60 sessions (days, not trades).
* **M** — Σ Close × Volume over the latest 60 sessions: approximate traded value in USD, not market capitalisation.

In [14]:
RFM_MULT = 2
obs_date = cal.max()                        # final common completed trading date
print('observation date:', obs_date.date())
assert obs_date < pd.Timestamp(END)

vol_r, close_r = vol_c[retained], close_c[retained]

# threshold for each share and day: 2 x mean volume of the PRECEDING 20 sessions (today excluded via shift(1))
base = vol_r.shift(1).rolling(RFM_BASE, min_periods=RFM_BASE).mean()
thr  = RFM_MULT * base
hv   = (vol_r > thr) & base.notna()         # strictly greater than twice the baseline

# 1.4.1 Recency: calendar days from latest high-volume day to the observation date
last_event = pd.Series({t: (hv.index[hv[t].values].max() if hv[t].any() else pd.NaT) for t in retained})
last_event = pd.to_datetime(last_event)
R = (obs_date - last_event).dt.days.astype('float')    # stays NaN if no event; never filled with 0

# 1.4.2 Frequency: number of high-volume days in the latest 60 sessions
F = hv.tail(RFM_WIN).sum()

# 1.4.3 Monetary: sum of Close x Volume over the latest 60 sessions (approx. traded value, USD)
M = (close_r * vol_r).tail(RFM_WIN).sum()

print('shares with no high-volume event (R missing):', int(R.isna().sum()))
print('shares with an event on the observation date (R = 0):', int((R == 0).sum()))
print('R range:', R.min(), '-', R.max(), '| F range:', int(F.min()), '-', int(F.max()))
print(f'M range: ${M.min():,.0f} - ${M.max():,.0f}')

observation date: 2024-12-31
shares with no high-volume event (R missing): 0
shares with an event on the observation date (R = 0): 0
R range: 4.0 - 222.0 | F range: 0 - 8
M range: $1,297,482,617 - $1,847,697,379,222


Edge-case checks: baseline history, 60-session window and no look-ahead.

In [15]:
# 1) every day in the 60-session window has a valid 20-session baseline
assert base.tail(RFM_WIN).notna().all().all(), 'insufficient history for the baseline in the window'
# 2) every share has at least 20 + 60 sessions
n_sessions = vol_r.notna().sum()
assert (n_sessions >= RFM_BASE + RFM_WIN).all()
print('history check passed: min sessions =', int(n_sessions.min()), '(needed', RFM_BASE + RFM_WIN, ')')

# 3) threshold alignment: the baseline for the last day must be the mean of the 20 sessions BEFORE it
t0 = 'AAPL' if 'AAPL' in retained else retained[0]
i = len(cal) - 1
assert np.isclose(base[t0].iloc[i], vol_r[t0].iloc[i - RFM_BASE:i].mean())
print('threshold alignment check passed: current day is excluded from its own baseline')

history check passed: min sessions = 1005 (needed 80 )
threshold alignment check passed: current day is excluded from its own baseline


**Event timing and boundary cases.** Is Recency driven by one market-wide day, and how are the edge cases handled?

In [16]:
# --- 1.4 event timing: is R driven by one market-wide day? ---
print('Most common latest high-volume dates:'); print(last_event.value_counts().head(5))
print(f"share of stocks with the same latest event date: {last_event.value_counts().iloc[0] / len(last_event):.1%}")
print('\nfraction of shares flagged on the busiest market days:'); print(hv.mean(axis=1).sort_values(ascending=False).head(5).round(3))

# --- 1.4 boundary cases ---
print('\nR == 0:', int((R == 0).sum()), '| missing R:', int(R.isna().sum()), '| F == 0:', int((F == 0).sum()))
print('share-days exactly at the threshold (not counted, rule is strictly greater):', int(((vol_r == thr) & base.notna()).sum().sum()))

Most common latest high-volume dates:
2024-12-20    353
2024-11-25     17
2024-11-06     11
2024-11-01      6
2024-11-15      6
Name: count, dtype: int64
share of stocks with the same latest event date: 72.6%

fraction of shares flagged on the busiest market days:
Date
2024-12-20    0.728
2023-12-15    0.656
2024-09-20    0.638
2022-12-16    0.582
2021-09-17    0.570
dtype: float64

R == 0: 0 | missing R: 0 | F == 0: 10
share-days exactly at the threshold (not counted, rule is strictly greater): 1
